# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. The Ukrainians w RE
2. Wielkowiejski Targ
3. Akira Kurosawa: Droga przez czas. Przegląd filmów w Kinie Kika
4. 26. Albertiana
5. Łukasz Wiśniewski Trio w Skawinie
6. Festiwal Misteria Paschalia 2026
7. Wystawa XII Konkursu Drzewek Emausowych
8. Cząstki kobiety. Wystawa Barbary Bazger
9. Alive Poster Festival – Post-Competition Exhibition
10. 20. Festiwal AfryKamera

Czas wykonania: 3.65s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. The Ukrainians w RE
2. Wielkowiejski Targ
3. Akira Kurosawa: Droga przez czas. Przegląd filmów w Kinie Kika
4. 26. Albertiana
5. Łukasz Wiśniewski Trio w Skawinie
6. Festiwal Misteria Paschalia 2026
7. Wystawa XII Konkursu Drzewek Emausowych
8. Cząstki kobiety. Wystawa Barbary Bazger
9. Alive Poster Festival – Post-Competition Exhibition
10. 20. Festiwal AfryKamera

Czas wykonania (wątki): 1.17s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 1.01s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [6]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact():
    return requests.get(CAT_API_URL).json().get("fact")


def run_sequential():
    print("Pobieranie sekwencyjne 20 faktów...")
    start = time.time()

    facts = []
    for _ in range(20):
        facts.append(get_cat_fact())

    end = time.time()

    print("\nPierwsze 5 faktów (sekwencyjnie):")
    for i, fact in enumerate(facts[:5], 1):
        print(f"{i}. {fact}")

    print(f"\nCzas sekwencyjny: {end - start:.4f} s")
    return end - start


def run_threaded():
    print("Pobieranie wielowątkowe 20 faktów...")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        facts = list(executor.map(lambda _: get_cat_fact(), range(20)))

    end = time.time()

    print("\nPierwsze 5 faktów (wątki):")
    for i, fact in enumerate(facts[:5], 1):
        print(f"{i}. {fact}")

    print(f"\nCzas wielowątkowy: {end - start:.4f} s")
    return end - start


seq_time = run_sequential()
thread_time = run_threaded()

print("\nPORÓWNANIE:")
print(f"Sekwencyjnie: {seq_time:.4f} s")
print(f"Wielowątkowo: {thread_time:.4f} s")

Pobieranie sekwencyjne 20 faktów...

Pierwsze 5 faktów (sekwencyjnie):
1. A female cat will be pregnant for approximately 9 weeks or between 62 and 65 days from conception to delivery.
2. A cat's normal temperature varies around 101 degrees Fahrenheit.
3. The biggest wildcat today is the Siberian Tiger. It can be more than 12 feet (3.6 m) long (about the size of a small car) and weigh up to 700 pounds (317 kg).
4. Neutering a male cat will, in almost all cases, stop him from spraying (territorial marking), fighting with other males (at least over females), as well as lengthen his life and improve its quality.
5. Cats' hearing stops at 65 khz (kilohertz); humans' hearing stops at 20 khz.

Czas sekwencyjny: 7.8360 s
Pobieranie wielowątkowe 20 faktów...

Pierwsze 5 faktów (wątki):
1. An adult lion's roar can be heard up to five miles (eight kilometers) away.
2. A cat has more bones than a human being; humans have 206 and the cat has 230 bones.
3. The name "jaguar" comes from a Native Amer

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [7]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

q = queue.Queue()
LICZBA_ELEMENTOW = 20

producer_done = threading.Event()
processed_count = 0
count_lock = threading.Lock()


def producer():
    for number in range(1, LICZBA_ELEMENTOW + 1):
        q.put(number)
        print(f"Producent dodał: {number}")
        time.sleep(0.05)
    producer_done.set()


def consumer(name, parity):
    global processed_count

    while True:
        try:
            number = q.get(timeout=0.1)
        except queue.Empty:
            with count_lock:
                if producer_done.is_set() and processed_count >= LICZBA_ELEMENTOW:
                    break
            continue

        if parity == "even" and number % 2 == 0:
            print(f"{name} pobrał parzystą: {number}")
            time.sleep(0.1)
            with count_lock:
                processed_count += 1
            q.task_done()

        elif parity == "odd" and number % 2 == 1:
            print(f"{name} pobrał nieparzystą: {number}")
            time.sleep(0.1)
            with count_lock:
                processed_count += 1
            q.task_done()

        else:
            q.put(number)
            q.task_done()
            time.sleep(0.01)


producer_thread = threading.Thread(target=producer)
consumer_even_thread = threading.Thread(target=consumer, args=("Konsument 1", "even"))
consumer_odd_thread = threading.Thread(target=consumer, args=("Konsument 2", "odd"))

producer_thread.start()
consumer_even_thread.start()
consumer_odd_thread.start()

producer_thread.join()
q.join()
consumer_even_thread.join()
consumer_odd_thread.join()

print("Program zakończył działanie.")

Producent dodał: 1
Konsument 2 pobrał nieparzystą: 1
Producent dodał: 2
Konsument 1 pobrał parzystą: 2
Producent dodał: 3
Konsument 2 pobrał nieparzystą: 3
Producent dodał: 4
Konsument 1 pobrał parzystą: 4
Producent dodał: 5
Konsument 2 pobrał nieparzystą: 5
Producent dodał: 6Konsument 1 pobrał parzystą: 6

Producent dodał: 7
Konsument 2 pobrał nieparzystą: 7
Producent dodał: 8
Konsument 1 pobrał parzystą: 8
Producent dodał: 9
Konsument 2 pobrał nieparzystą: 9
Producent dodał: 10
Konsument 1 pobrał parzystą: 10
Producent dodał: 11Konsument 2 pobrał nieparzystą: 11

Producent dodał: 12
Konsument 1 pobrał parzystą: 12
Producent dodał: 13
Konsument 2 pobrał nieparzystą: 13
Producent dodał: 14
Konsument 1 pobrał parzystą: 14
Producent dodał: 15
Konsument 2 pobrał nieparzystą: 15
Producent dodał: 16Konsument 1 pobrał parzystą: 16

Producent dodał: 17
Konsument 2 pobrał nieparzystą: 17
Producent dodał: 18
Konsument 1 pobrał parzystą: 18
Producent dodał: 19
Konsument 2 pobrał nieparzystą: 19


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [10]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum


if __name__ == "__main__":
    start = time.time()

    numbers = list(range(1, 10001))
    cores = multiprocessing.cpu_count()

    print(f"Liczba dostępnych rdzeni: {cores}")
    print("Rozpoczynam obliczenia...")

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)

    end = time.time()

    print(f"Obliczono wyniki dla {len(results)} liczb.")
    print("Pierwsze 5 wyników:")
    for i, result in enumerate(results[:5], 1):
        print(f"{i}. {result}")

    print(f"\nCzas wykonania: {end - start:.2f} s")

Liczba dostępnych rdzeni: 12
Rozpoczynam obliczenia...
Obliczono wyniki dla 10000 liczb.
Pierwsze 5 wyników:
1. 100
2. 2535301200456458802993406410750
3. 773066281098016996554691694648431909053161283000
4. 2142584059011987034055949456454883470029603991710390447068500
5. 9860761315262647567646607066034827870915080438862787559628486633300780

Czas wykonania: 0.32 s
